In [1]:
import time
t1 = time.time()

import math
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [2]:
##### load data #####

hhvv_HLdata_path = "/root/data/mucollider/two_boosted/scan_kappa3_hhvv_HL.h5"
hhvv_HLdata_file = h5py.File(hhvv_HLdata_path, 'r')
print(hhvv_HLdata_file.keys())

hhmumu_HLdata_path = "/root/data/mucollider/two_boosted/scan_kappa3_hhmumu_HL.h5"
hhmumu_HLdata_file = h5py.File(hhmumu_HLdata_path, 'r')
print(hhmumu_HLdata_file.keys())

<KeysViewHDF5 ['BTag_leading', 'BTag_subleading', 'H_0', 'H_1', 'LB_1', 'M_JJ', 'S_0', 'S_1', 'X_HH', 'dEta_JJ', 'm_J_leading', 'm_J_subleading', 'missET', 'pT_JJ', 'pt_J_leading', 'pt_J_subleading', 'target_kappa_3']>
<KeysViewHDF5 ['BTag_leading', 'BTag_subleading', 'H_0', 'H_1', 'LB_1', 'M_JJ', 'S_0', 'S_1', 'X_HH', 'dEta_JJ', 'm_J_leading', 'm_J_subleading', 'missET', 'pT_JJ', 'pt_J_leading', 'pt_J_subleading', 'target_kappa_3']>


In [3]:
##### include data #####

HLfeatures = ['BTag_leading', 'BTag_subleading', 'H_0', 'H_1', 'LB_1', 'M_JJ', 'S_0', 'S_1', 'X_HH', 'dEta_JJ', 'm_J_leading', 'm_J_subleading', 'missET', 'pT_JJ', 'pt_J_leading', 'pt_J_subleading']

##### select BTag work point at WP70 (BitNumber 1) #####

hhvv_BTag_leading = ((hhvv_HLdata_file['BTag_leading'][:])&2)
hhvv_BTag_subleading = ((hhvv_HLdata_file['BTag_subleading'][:])&2)

hhmumu_BTag_leading = ((hhmumu_HLdata_file['BTag_leading'][:])&2)
hhmumu_BTag_subleading = ((hhmumu_HLdata_file['BTag_subleading'][:])&2)

##### include other high level features data #####

hhvv_HLdata = []
for feature in HLfeatures:
    if feature=='BTag_leading': hhvv_HLdata.append(hhvv_BTag_leading)
    elif feature=='BTag_subleading': hhvv_HLdata.append(hhvv_BTag_subleading)
    else: hhvv_HLdata.append(hhvv_HLdata_file[feature][:])
hhvv_HLdata = np.array(hhvv_HLdata).T
print('shape of high level data:', hhvv_HLdata.shape)

hhmumu_HLdata = []
for feature in HLfeatures:
    if feature=='BTag_leading': hhmumu_HLdata.append(hhmumu_BTag_leading)
    elif feature=='BTag_subleading': hhmumu_HLdata.append(hhmumu_BTag_subleading)
    else: hhmumu_HLdata.append(hhmumu_HLdata_file[feature][:])
hhmumu_HLdata = np.array(hhmumu_HLdata).T
print('shape of high level data:', hhmumu_HLdata.shape)

shape of high level data: (321016, 16)
shape of high level data: (314674, 16)


In [4]:
##### define the event information #####

BR_hbb = 0.5824
Luminosity = 1000   ### unit: fb^-1
simulation_num = 100000
kappa3 = [0.2, 0.4, 0.6, 0.8, 0.9, 1.0, 1.1, 1.2, 1.4, 1.6, 1.8]

hhvv_Xsection_NoDecay = [1.23322e-01, 1.016366e-01, 8.33476424e-02, 6.81961e-02, 6.2211286e-02, 5.648398e-02, 5.187586e-02, 4.801255e-02, 4.246736e-02, 4.038406e-02, 4.1308219e-02]     ### hhvv unit: fb
hhvv_Xsection = [hhvv_Xsection_NoDecay[i]*BR_hbb*BR_hbb for i in range(11)]
hhvv_event_num = [31536, 31077, 30923, 30485, 30011, 29716, 29039, 28528, 27438, 26495, 25768]

hhmumu_Xsection_NoDecay = [1.068573e-2, 8.89267e-3, 7.29573e-3, 5.892737e-3, 5.2836183e-3, 4.68848e-3, 4.1895e-3, 3.7137245e-3, 2.9218193e-3, 2.3228e-3, 1.9414667e-3]     ### hhmumu unit: fb
hhmumu_Xsection = [hhmumu_Xsection_NoDecay[i]*BR_hbb*BR_hbb for i in range(11)]
hhmumu_event_num = [30469, 30020, 29858, 29497, 29190, 28845, 28653, 28343, 27442, 26613, 25744]

In [5]:
##### select the training data #####

hhvv_04data = hhmumu_HLdata[31536:31536+31077]
hhvv_16data = hhmumu_HLdata[268753:295248]
print("kappa3 = 0.4, datashape:", hhvv_04data.shape)
print("kappa3 = 1.6, datashape:", hhvv_16data.shape)

target_04 = np.zeros(31077)
target_16 = np.ones(26495)

X = np.concatenate((hhvv_04data, hhvv_16data))
Y = np.concatenate((target_04, target_16))

X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, random_state=2330, shuffle=True)
print("\nAfter training validation data split:")
print(f"Training features data shape: {X_train.shape}, Validation features data shape: {X_val.shape}")
print(f"Training targets data shape: {Y_train.shape}, Validation targets data shape: {Y_val.shape}")

kappa3 = 0.4, datashape: (31077, 16)
kappa3 = 1.6, datashape: (26495, 16)

After training validation data split:
Training features data shape: (46057, 16), Validation features data shape: (11515, 16)
Training targets data shape: (46057,), Validation targets data shape: (11515,)


In [6]:
##### train the model #####

xgb_0416 = XGBClassifier(n_estimators=100, learning_rate= 0.3)
xgb_0416.fit(X_train, Y_train)

print("Time:{:^8.4f}(s)".format(time.time()-t1))

Time:125.1077(s)


In [7]:
Y_pred = xgb_0416.predict(X_val)
accuracy = accuracy_score(Y_val, Y_pred)
print(f"模型測試集準確率: {accuracy * 100:.2f}%")

模型測試集準確率: 67.48%


In [ ]:
????????????????? 67 ?????????????????????